# Post-hoc analysis of AF3 batch data
- Discover completed AF3 jobs and read-out JSON files storing confidence scores (pLDDT, pTM, ipTM)
- Collect confidence scores from all completed jobs and save a batch summary CSV file containing top-rank model scores of each job
- Save a detailed "per-sample" lookup CSV file containing scores from all predicted samples of all jobs (5 samples per seed)

In [ ]:
from pathlib import Path
import csv
import json
import math
import re


# ============================================================
# Settings
# ============================================================

OUTPUT_DIR = Path("./")
BATCH_SUMMARY_PATH = OUTPUT_DIR / "batch_summary.csv"
PER_SAMPLE_PATH = OUTPUT_DIR / "overview_per-sample.csv"


# ============================================================
# Helper functions
# ============================================================

def safe_float(value):
    """Return a float if possible, otherwise None."""
    if value is None:
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def ranking_sort_value(row):
    """Sort missing/invalid ranking scores to the bottom."""
    value = safe_float(row.get("ranking_score"))
    if value is None or math.isnan(value):
        return float("-inf")
    return value


def find_jobs(output_dir):
    """
    Discover completed AF3 job directories.

    A completed job is identified by:
        <jobname>/<jobname>_summary_confidences.json
    """
    jobs = []

    for job_dir in sorted(output_dir.iterdir()):
        if not job_dir.is_dir():
            continue
        jobname = job_dir.name
        summary_file = job_dir / f"{jobname}_summary_confidences.json"
        if summary_file.exists():
            jobs.append({
                "jobname": jobname,
                "job_dir": job_dir,
            })
    return jobs

# ============================================================
# Discover jobs
# ============================================================

jobs = find_jobs(OUTPUT_DIR)
print(f"Found {len(jobs)} completed AF3 jobs in:")
print(OUTPUT_DIR)

# ============================================================
# 1. Build batch_summary.csv
# ============================================================

batch_rows = []

for job in jobs:
    jobname = job["jobname"]
    job_dir = job["job_dir"]

    summary_path = job_dir / f"{jobname}_summary_confidences.json"
    confidence_path = job_dir / f"{jobname}_confidences.json"

    with summary_path.open() as f:
        summ = json.load(f)

    # AlphaFold output may already contain mean_plddt
    mean_plddt = summ.get("mean_plddt")

    # Fallback: calculate it from detailed confidence JSON
    if mean_plddt is None and confidence_path.exists():
        with confidence_path.open() as f:
            conf = json.load(f)

        plddts = conf.get(
            "atom_plddts",
            conf.get("token_plddts", [])
        )

        if plddts:
            mean_plddt = sum(plddts) / len(plddts)

    batch_rows.append({
        "jobname": jobname,
        "mean_plddt": mean_plddt,
        "ptm": summ.get("ptm"),
        "iptm": summ.get("iptm"),
        "ranking_score": summ.get("ranking_score"),
        "fraction_disordered": summ.get("fraction_disordered"),
        "has_clash": summ.get("has_clash"),
    })


# Sort best ranking score first
batch_rows.sort(
    key=ranking_sort_value,
    reverse=True,
)

batch_fields = [
    "jobname",
    "mean_plddt",
    "ptm",
    "iptm",
    "ranking_score",
    "fraction_disordered",
    "has_clash",
]

with BATCH_SUMMARY_PATH.open("w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=batch_fields,
    )
    writer.writeheader()
    writer.writerows(batch_rows)

print(
    f"\nWrote {len(batch_rows)} jobs -> "
    f"{BATCH_SUMMARY_PATH}"
)


# ============================================================
# 2. Build overview_per-sample.csv
# ============================================================

sample_rows = []

sample_dir_pattern = re.compile(
    r"seed-(\d+)_sample-(\d+)$"
)

for job in jobs:
    jobname = job["jobname"]
    job_dir = job["job_dir"]

    # Look for:
    # seed-X_sample-Y/*_summary_confidences.json
    for summary_path in sorted(
        job_dir.glob(
            "seed-*_sample-*/*_summary_confidences.json"
        )
    ):
        sample_dir = summary_path.parent
        match = sample_dir_pattern.match(sample_dir.name)

        if match is None:
            print(
                f"Warning: could not parse seed/sample from "
                f"{sample_dir.name}"
            )
            continue

        seed = int(match.group(1))
        sample = int(match.group(2))

        with summary_path.open() as f:
            summ = json.load(f)

        # chain_ids may contain repeated entries.
        # Remove duplicates while preserving order.
        unique_chains = list(
            dict.fromkeys(
                summ.get("chain_ids", [])
            )
        )

        chain_ptm = summ.get("chain_ptm", [])
        chain_iptm = summ.get("chain_iptm", [])

        row = {
            "jobname": jobname,
            "seed": seed,
            "sample": sample,
            "ptm": summ.get("ptm"),
            "iptm": summ.get("iptm"),
            "ranking_score": summ.get("ranking_score"),
            "fraction_disordered": summ.get(
                "fraction_disordered"
            ),
            "has_clash": summ.get("has_clash"),
            "chain_ids": ",".join(unique_chains),
            "chain_pair_iptm": json.dumps(
                summ.get("chain_pair_iptm")
            ),
            "chain_pair_pae_min": json.dumps(
                summ.get("chain_pair_pae_min")
            ),
        }

        # Add one pTM and ipTM column per chain
        for chain, value in zip(
            unique_chains,
            chain_ptm,
        ):
            row[f"ptm_chain_{chain}"] = value

        for chain, value in zip(
            unique_chains,
            chain_iptm,
        ):
            row[f"iptm_chain_{chain}"] = value

        sample_rows.append(row)


# Dynamic columns because different jobs can contain
# different numbers of chains/ligands.
fixed_cols = [
    "jobname",
    "seed",
    "sample",
    "ptm",
    "iptm",
    "ranking_score",
    "fraction_disordered",
    "has_clash",
    "chain_ids",
]

chain_cols = sorted({
    key
    for row in sample_rows
    for key in row
    if key.startswith(
        ("ptm_chain_", "iptm_chain_")
    )
})

extra_cols = [
    "chain_pair_iptm",
    "chain_pair_pae_min",
]

sample_fields = (
    fixed_cols
    + chain_cols
    + extra_cols
)

sample_rows.sort(
    key=lambda row: (
        row["jobname"],
        row["seed"],
        row["sample"],
    )
)

with PER_SAMPLE_PATH.open("w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=sample_fields,
    )
    writer.writeheader()
    writer.writerows(sample_rows)

print(
    f"Wrote {len(sample_rows)} sample rows "
    f"from {len(jobs)} jobs -> "
    f"{PER_SAMPLE_PATH}"
)